# Tutorial 7c — Two-model coupling III: asking questions of a coupled model

Estimated time: 25-30 minutes. Needs PyMC. SBI optional (Step 5 uses it if present).

> **Module 7, part 3 of 3.** 7a said what a coupling is; 7b sampled it and showed how to
> tell whether the sampler worked. This part is the payoff.
>
> | | Question | Needs |
> |---|---|---|
> | 7a | What does it mean to couple two models? | nothing extra |
> | 7b | How is the coupled distribution sampled, and can I trust it? | PyMC from Step 3 |
> | **7c (you are here)** | What can I ask it? | PyMC (SBI optional) |

## The question this part answers

You have several models of the same system. You can **measure** a couple of things — usually
whatever a microscope can see. Everything else is hidden: rate constants, diffusion
coefficients, timescales, geometry.

> **Can I take what I measured in one model's output, and use it to infer a parameter that
> lives in a completely different model?**

Yes. That is what this notebook does, and it is the exact shape of the inference in the paper
this repository reproduces:

$$\Pr\big(\underbrace{\text{DL}, \text{Dep}, t_{KS}, R_{KS}, \text{Diff}, P_{\text{off}}}_{\text{six hidden parameters, in three different models}} \;\big|\; \underbrace{\text{Phos}_{\text{obs}}, \text{Rg}_{\text{obs}}}_{\text{two things a microscope measured}}\big)$$

Two numbers in; six out; and the six live in models that were built separately, by different
people, in different software.

## What you'll be able to do afterwards

- **Run that query**, on a chain of models you fit yourself.
- **Recover a known truth** through two model boundaries, and check it.
- **Read an identifiability ridge** — where your data pins a *combination* of parameters but
  not the parameters themselves. The paper's headline result is one of these.
- **Say when conditioning will not reach a variable**, before wasting a run on it.
- **Choose between the two surrogate backends** for a given problem.

## How you'll know you got it

Given a chain of models and a measurement at one end, you can predict which parameters will
be informed, which will be informed only in combination, and which will not move at all —
then check yourself against the run.


## The science: three models, two measurements, six unknowns

The system is the **immune synapse** — the contact between a T cell and the surface it is
inspecting. Within seconds of contact, T-cell receptors (TCRs) at the contact become
phosphorylated (chemically tagged, the first step of "this cell is activated"), and they do
so in a *ring at the periphery* rather than uniformly. Explaining that ring is what the
paper set out to do, and no single model could.

Three partial models, each built and validated on its own:

| Model | What it describes | Its hidden parameters | Its output |
|---|---|---|---|
| **Kinetic segregation (KS)** | The membrane is squeezed at the contact; the bulky phosphatase CD45 is physically excluded from the tightest region | membrane rigidity `R`, contact age `t` | depletion width `Dep` — how far CD45 is pushed back |
| **Lck activation** | Active Lck kinase spreads outward from where CD45 is absent, decaying as it goes | diffusion coefficient `Diff`, deactivation rate `P_off` | decay length `DL` — how far Lck activity reaches |
| **TCR phosphorylation** | TCRs get phosphorylated where active Lck is present and CD45 is not | — (it consumes the other two) | `Phos` (what fraction are phosphorylated) and `Rg` (how peripheral the pattern is) |

*(CD45 is a phosphatase — it removes phosphate groups, so it switches TCR signalling **off**.
Lck is a kinase — it adds them, switching signalling **on**. The competition between them is
the whole story.)*

**What a microscope can see:** `Phos` and `Rg`, by counting and locating phosphorylated TCRs
in an image. Those are the two orange numbers in the paper's figure.

**What it cannot see:** every parameter in the first two columns. Membrane rigidity, contact
age, Lck's diffusion coefficient and deactivation rate are not visible in the image at all.

So the question is forced on you: *given only the two things I can measure, what must the
six things I cannot measure have been?* That is a conditional inference across model
boundaries, and it is the reason to build a joint model rather than three separate ones.


In [ ]:
# Cross-platform setup — the same opening cell as every other tutorial.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap  # noqa: E402

root = bootstrap()
ROOT = root
print("Repo root:", root)

try:
    import pymc  # noqa: F401
    PYMC_AVAILABLE = True
    print("PyMC found — every step below will run.")
except ImportError:
    PYMC_AVAILABLE = False
    print("=" * 72)
    print("PREFLIGHT: PyMC is missing, so the sampling steps will skip.")
    print("  conda env create -f environment-all.yml")
    print("  conda activate py312_bayesmm_all")
    print("The explanations still read fine; only the cells that sample need it.")
    print("=" * 72)


## Step 1: conditioning, in its simplest possible form

Here is the capability 7b did not have. It could draw the joint distribution of a coupled
model; it had no way to say **"I measured this one"**.

The spec now takes an `observed` block:

```json
{"observed": {"y": 3.0}}
```

An observed variable is **clamped** — not drawn, not proposed, held exactly at its value
while every factor that mentions it is evaluated there. It leaves the sample space, so the
sampler works in one fewer dimension.

For the two-variable model this also has an answer on paper, so once again we can check
rather than trust.


In [ ]:
import numpy as np

if not PYMC_AVAILABLE:
    print("Step 1 SKIPPED — needs PyMC (see preflight above).")
    nuts_s = cs = None
else:
    from bayesian_metamodeling.meta.compiler import compile_metamodel
    from bayesian_metamodeling.meta.ir import (
        CouplingFactorIR, MetamodelIR, PriorFactorIR, VariableIR,
    )
    from bayesian_metamodeling.meta.nuts_sampling import sample_nuts

    # The same two-variable coupled model 7b checked against pen and paper.
    mx, sx, my, sy, alpha, beta, sigma = 1.0, 0.5, 0.0, 2.0, 1.5, -0.4, 0.3
    pair_factors = [
        PriorFactorIR(variable="x", distribution={"kind": "normal", "loc": mx, "scale": sx}),
        PriorFactorIR(variable="y", distribution={"kind": "normal", "loc": my, "scale": sy}),
        CouplingFactorIR(coupling_type="gaussian_link", source="x", target="y",
                         transform={"kind": "affine", "alpha": alpha, "beta": beta},
                         sigma=sigma),
    ]
    pair = MetamodelIR(name="pair", variables=[VariableIR(name="x"), VariableIR(name="y")],
                       factors=pair_factors)
    nuts_s, _ = sample_nuts(compile_metamodel(pair), draws=3000, tune=1000, chains=2, seed=11)

    y_measured = 3.0
    conditioned = MetamodelIR(name="pair_conditioned",
                              variables=[VariableIR(name="x"), VariableIR(name="y")],
                              factors=pair_factors, observed={"y": y_measured})
    cs, cd = sample_nuts(compile_metamodel(conditioned), draws=3000, tune=1000, chains=2, seed=3)

    # p(x | y) for a normal prior plus a linear-Gaussian coupling, by completing the square.
    prec = 1/sx**2 + alpha**2/sigma**2
    post_mean = (mx/sx**2 + alpha*(y_measured - beta)/sigma**2) / prec
    post_sd = 1/np.sqrt(prec)

    print(f"before measuring:   x = {nuts_s['x'].mean():+.3f} +- {nuts_s['x'].std():.3f}")
    print(f"after y = {y_measured}:      x = {cs['x'].mean():+.3f} +- {cs['x'].std():.3f}")
    print(f"on paper:           x = {post_mean:+.3f} +- {post_sd:.3f}")
    print()
    print(f"y held exactly?  all draws == {y_measured}: {bool(np.all(cs['y'] == y_measured))}")
    print(f"variables actually sampled: {cd['free_variables']}  (y is not among them)")


**Two things happened, and both are what "inference" means.** `x` *moved* — the
measurement of `y` told us something about it — and `x` got *narrower*, because a
measurement is information. The paper-and-pencil answer confirms both.

Note this is only possible because a coupling is written as a **factor** rather than as a
generative step (T7, Step B). A factor is symmetric in its variables, so evidence can enter
at either end and travel in either direction. If the coupling had been written as "`y` is
produced from `x`", information could only ever have flowed one way.


## Step 2: the real query — two measurements, three models, four unknowns

Now the thing this notebook exists for. We build a small honest analogue of the paper's
metamodel and run its actual query.

**The chain**, with each model fitted separately from its own sweep, exactly as T5 did:

```
   log_diff, log_poff  ──[ Lck model ]──▶  log_decay ──┐
                                                       ├──[ pTCR model ]──▶  phos, rg_ratio
                                         depletion ────┘
```

**The physics that matters**, and it is worth a moment because Step 3 turns on it. A
molecule that diffuses with coefficient `D` while being deactivated at rate `k_off` travels
a characteristic distance before it switches off:

$$\lambda = \sqrt{D / k_{\text{off}}}$$

Take logs and that becomes a straight line, which is convenient because our surrogate
backend (`pymc_gp`) fits linear models:

$$\log \lambda = \tfrac{1}{2}\log D - \tfrac{1}{2}\log k_{\text{off}}$$

Read that formula and predict, before running anything: **if you only ever observe `λ`, can
you recover `D`? Can you recover `k_off`? Can you recover anything about them?**

We pick a true hidden state, compute what a microscope would have seen, then hand the
notebook *only those two numbers* and ask it to find its way back.


In [ ]:
if not PYMC_AVAILABLE:
    print("Step 2 SKIPPED — needs PyMC.")
    chain_post = None
else:
    from bayesian_metamodeling.meta.ir import SurrogateLikelihoodFactorIR
    from bayesian_metamodeling.surrogates.backends import fit_backend_model

    _rng = np.random.default_rng(0)

    def _fit(cols, truth, names, out, noise, seed):
        """Fit one partial model's surrogate from a noisy sweep of it."""
        return fit_backend_model(
            backend="pymc_gp", x=np.column_stack(cols),
            y=(truth + _rng.normal(0, noise, len(truth))).reshape(-1, 1),
            input_names=names, output_names=[out],
            backend_config={"draws": 400, "tune": 400, "chains": 1, "target_accept": 0.9},
            seed=seed)

    # --- Lck model: log_decay = 0.5*log_diff - 0.5*log_poff  (i.e. lambda = sqrt(D/k)) ---
    _lD = _rng.uniform(9, 14, 200)          # log diffusion coefficient
    _lK = _rng.uniform(-2, 2, 200)          # log deactivation rate
    Lck = _fit([_lD, _lK], 0.5*_lD - 0.5*_lK, ["log_diff", "log_poff"], "log_decay", 0.02, 1)

    # --- pTCR model: two readouts, both driven by decay length and depletion width ---
    _ld = _rng.uniform(4, 8, 200)
    _dep = _rng.uniform(50, 400, 200)
    A, B, C, E = 0.020, 0.00030, 0.150, 0.00100
    pT_phos = _fit([_ld, _dep], A*_ld + B*_dep, ["log_decay", "depletion"], "phos", 0.004, 2)
    pT_rg   = _fit([_ld, _dep], C*_ld + E*_dep, ["log_decay", "depletion"], "rg_ratio", 0.004, 3)

    # --- the true hidden state, and what a microscope would therefore have measured ---
    TRUE_LOG_DECAY, TRUE_DEPLETION = 6.0, 220.0
    phos_obs = round(A*TRUE_LOG_DECAY + B*TRUE_DEPLETION, 4)
    rg_obs   = round(C*TRUE_LOG_DECAY + E*TRUE_DEPLETION, 4)
    print(f"true hidden state : log_decay = {TRUE_LOG_DECAY}, depletion = {TRUE_DEPLETION}")
    print(f"microscope sees   : phos = {phos_obs}, rg_ratio = {rg_obs}")
    print("Everything else below is inferred from those two numbers alone.\n")

    def _chain(observed):
        return MetamodelIR(
            name="immune_synapse_analogue",
            variables=[VariableIR(name=n) for n in
                       ("log_diff", "log_poff", "log_decay", "depletion", "phos", "rg_ratio")],
            factors=[
                PriorFactorIR(variable="log_diff", distribution={"kind": "normal", "loc": 11.5, "scale": 1.2}),
                PriorFactorIR(variable="log_poff", distribution={"kind": "normal", "loc": 0.0, "scale": 1.2}),
                PriorFactorIR(variable="log_decay", distribution={"kind": "normal", "loc": 6.0, "scale": 1.5}),
                PriorFactorIR(variable="depletion", distribution={"kind": "normal", "loc": 225.0, "scale": 100.0}),
                PriorFactorIR(variable="phos", distribution={"kind": "normal", "loc": 0.15, "scale": 0.10}),
                PriorFactorIR(variable="rg_ratio", distribution={"kind": "normal", "loc": 1.1, "scale": 0.4}),
                SurrogateLikelihoodFactorIR(surrogate_ref="Lck", inputs=["log_diff", "log_poff"],
                                            outputs=["log_decay"]),
                SurrogateLikelihoodFactorIR(surrogate_ref="pT_phos", inputs=["log_decay", "depletion"],
                                            outputs=["phos"]),
                SurrogateLikelihoodFactorIR(surrogate_ref="pT_rg", inputs=["log_decay", "depletion"],
                                            outputs=["rg_ratio"]),
            ],
            observed=observed)

    _surr = {"Lck": Lck, "pT_phos": pT_phos, "pT_rg": pT_rg}
    chain_prior, _ = sample_nuts(compile_metamodel(_chain({})), draws=3000, tune=1500,
                                 chains=2, seed=5, surrogates=_surr)
    chain_post, chain_d = sample_nuts(
        compile_metamodel(_chain({"phos": phos_obs, "rg_ratio": rg_obs})),
        draws=3000, tune=1500, chains=2, seed=5, surrogates=_surr)

    print(f"{'hidden quantity':<14}{'before':>20}{'after measuring':>22}{'truth':>9}")
    for v, truth in (("log_decay", TRUE_LOG_DECAY), ("depletion", TRUE_DEPLETION),
                     ("log_diff", None), ("log_poff", None)):
        a = np.asarray(chain_prior[v], float).ravel()
        b = np.asarray(chain_post[v], float).ravel()
        t = f"{truth}" if truth is not None else "—"
        print(f"{v:<14}{a.mean():>11.3f} +-{a.std():<7.3f}{b.mean():>13.3f} +-{b.std():<7.3f}{t:>9}")
    print(f"\nr-hat={max(chain_d['r_hat'].values()):.4f}  divergences={chain_d['divergences']}  "
          f"worst ESS={min(chain_d['ess'].values()):.0f}")


**`log_decay` and `depletion` are recovered**, to well within their uncertainty, from two
numbers measured at the far end of the chain. Neither was measured. `depletion` lives in a
model that the measurement does not even touch directly — it reaches it through the pTCR
surrogate.

**But look at `log_diff` and `log_poff`.** Their spreads barely moved from their priors. The
measurement seems to have told us almost nothing about the two parameters we probably cared
about most.

That is not a failure of the sampler, and Step 5 is about why.


## Step 3: the ridge — when the data pins a *combination*, not the parameters

Go back to the physics:

$$\log \lambda = \tfrac{1}{2}\log D - \tfrac{1}{2}\log k_{\text{off}}$$

The measurement constrains `λ`. But `λ` depends on `D` and `k_off` **only through their
difference in logs** — that is, only through the ratio `D / k_off`. Double both and `λ` does
not change at all. No amount of data of this kind can separate them.

This is called an **identifiability** problem, and it is not a defect: it is a true
statement about what your experiment can and cannot see. The honest response is to report
the combination that *is* determined, and say plainly that the individual values are not.

The cell below measures exactly that.


In [ ]:
if not PYMC_AVAILABLE or chain_post is None:
    print("Step 3 SKIPPED — needs Step 2.")
else:
    import matplotlib.pyplot as plt

    D = np.asarray(chain_post["log_diff"], float).ravel()
    K = np.asarray(chain_post["log_poff"], float).ravel()
    D0 = np.asarray(chain_prior["log_diff"], float).ravel()
    K0 = np.asarray(chain_prior["log_poff"], float).ravel()

    print(f"  sd(log_diff)            = {D.std():.3f}   (prior sd 1.2 — barely informed)")
    print(f"  sd(log_poff)            = {K.std():.3f}   (prior sd 1.2 — barely informed)")
    print(f"  sd(log_diff - log_poff) = {np.std(D - K):.3f}   <- the combination IS informed")
    print(f"  correlation             = {np.corrcoef(D, K)[0, 1]:+.3f}")
    print()
    print(f"  implied log_decay = (log_diff - log_poff)/2 = {np.mean(D - K)/2:.3f} "
          f"+- {np.std(D - K)/2:.3f}   (truth {TRUE_LOG_DECAY})")

    fig, ax = plt.subplots(figsize=(6.2, 5.4), constrained_layout=True)
    ax.scatter(D0, K0, s=5, alpha=0.15, color="tab:gray", label="before measuring")
    ax.scatter(D, K, s=5, alpha=0.30, color="tab:red", label="after measuring")
    _x = np.linspace(D.min() - 0.3, D.max() + 0.3, 50)
    ax.plot(_x, _x - 2*TRUE_LOG_DECAY, "k--", lw=2,
            label=r"$\log D - \log k_{off} = 2\log\lambda$")
    ax.set_xlabel("log Diff  (diffusion coefficient)")
    ax.set_ylabel("log P_off  (deactivation rate)")
    ax.set_title("The posterior is a ridge, not a blob")
    ax.legend(loc="upper left", fontsize=9)
    plt.show()


**That diagonal band is the result.** Every point on it explains the measurement equally
well. The data has pinned the *ratio* and left the position along the ridge to the priors.

This is the paper's headline finding, and it is worth seeing that it was a discovery, not a
nuisance: the authors report a constraint of the form **Diff = C · P_off** — a straight line
in exactly this plane (their Figure 7G) — as the relationship required to produce the
peripheral phosphorylation ring seen in real cells. Their metamodel could not say what the
diffusion coefficient *was*; it could say what it had to be **relative to** the deactivation
rate, and that was the biologically meaningful answer.

Three practical lessons a student should leave with:

1. **Report the combination.** "log_decay = 6.0 ± 0.1" is a real result. "log_diff = 11.7 ±
   0.9" is mostly a restatement of your prior, and presenting it as a finding would be
   misleading.
2. **A ridge is exactly the geometry a random walk handles worst** (7b, Step 4) — it must propose
   diagonal moves it never thinks to make. This is why 7b's efficiency gap matters here rather than
   being a benchmarking curiosity.
3. **Non-identifiability tells you what experiment to do next.** If you need `Diff` itself,
   no amount of this measurement will give it — you need a measurement that sees `D` and
   `k_off` differently, such as a time-resolved one.


## Step 4: when conditioning will *not* reach a variable

Conditioning travels along factors. If nothing connects your measurement to the variable you
care about, nothing happens — and the run will not warn you, because a wide posterior is not
an error.

This is not hypothetical in this repository. The real TCR metamodel in
`projects/tcr_signaling` has four partial models, and the output of the kinetic-segregation
model — `depletion_width_nm` — is consumed by **nothing**. No other surrogate takes it as an
input, and no coupling references it.

The consequence is immediate: measuring the pTCR readout informs the variables that have a
path to it, and leaves the KS model's parameters (`rigidity_kT_nm2`, `time_sec`) sitting at
their priors, no matter how long you sample.

*(In the published metamodel, KS's depletion width **is** an input to the pTCR model — that
is the coupling that makes membrane mechanics matter for signalling. Reconnecting it here
would mean refitting the pTCR surrogate with depletion width among its inputs. It is a
worthwhile exercise and a real gap, not a bug in the sampler.)*

**A terminal output is not automatically a problem.** `ptcr_fraction` also feeds nothing —
but it is what the microscope measures, so it is the *entry point* for evidence rather than a
dead end. The question to ask per model is sharper than "does my output feed something":

> **Can evidence reach this model's parameters at all** — either because one of its outputs
> is measured directly, or because one of its outputs feeds something else that eventually is?

The cell below asks exactly that, mechanically, for all four models.


In [ ]:
import json as _json

_specs = ROOT.parent.parent / "projects" / "tcr_signaling" / "specs"
if not _specs.is_dir():
    _specs = ROOT / "projects" / "tcr_signaling" / "specs"

if not _specs.is_dir():
    print("tcr_signaling submodule not checked out — skipping this inspection.")
    print("The point stands: check whether a path of factors connects your measurement")
    print("to the variable you care about, before running anything.")
else:
    _models = {}
    for _f in sorted(_specs.glob("model.*.json")):
        _d = _json.loads(_f.read_text())
        _models[_f.stem.replace("model.", "")] = (
            [i["name"] for i in _d["io_schema"]["inputs"]],
            [o["name"] for o in _d["io_schema"]["outputs"]])
    _mm = _json.loads((_specs / "metamodel.tcr_signaling.json").read_text())
    _coupled = {c["source"] for c in _mm["couplings"]}

    # What you would actually measure in this system — the microscope readouts.
    MEASURABLE = {"ptcr_fraction", "ptcr_density"}

    print("For each model: can a measurement ever reach its parameters?\n")
    for _name, (_ins, _outs) in _models.items():
        _routes = []
        for _o in _outs:
            _consumers = [m for m, (i2, _) in _models.items() if _o in i2 and m != _name]
            if _consumers:
                _routes.append(f"{_o} -> {_consumers}")
            elif _o in _coupled:
                _routes.append(f"{_o} -> coupled onward")
            elif _o in MEASURABLE:
                _routes.append(f"{_o} = measured directly")
        _reachable = bool(_routes)
        _mark = "reachable" if _reachable else "UNREACHABLE"
        print(f"  [{_mark:^11}] {_name}")
        for _r in _routes:
            print(f"                  via {_r}")
        if not _reachable:
            _dead = [o for o in _outs if o not in MEASURABLE]
            print(f"                  its outputs {_dead} feed nothing and are not measured,")
            print(f"                  so no data can inform {_ins[:3]}...")


Read the dead ends as a to-do list for the modelling, not as a limitation of the method.
A variable with no path to your data is one your data cannot speak to — the fix is a
coupling or a shared variable, decided on scientific grounds.

**The general rule**, and the answer to "can I condition any variable on any other?":

> Yes — **provided a path of factors connects them.** Direction does not matter (evidence
> flows upstream and downstream alike), model boundaries do not matter, and the number of
> hops does not matter. What matters is that the graph is connected and that the surrogates
> along the path are actually informative.


## Step 5: PyMC or SBI — and can I have both?

A fair question, since T5 and T6 presented `pymc_gp` and `sbi_npe` as two backends behind one
interface. For *fitting a surrogate* they are close to interchangeable. For the inference in
this notebook they are **not**, and the reason is structural rather than a matter of quality.

| | `pymc_gp` | `sbi_npe` |
|---|---|---|
| what it fits | Bayesian **linear** regression (despite the name — see T5) | a neural density estimator |
| flexibility | low: straight lines and planes only | high: learns curved, multi-modal shapes |
| its density is | a formula you can write down and differentiate | a neural network's output |
| usable by `--method nuts` | **yes** | no — falls back to 7b's random walk |
| honest about extrapolation | no: predicts confidently far outside its training data | somewhat better, but do not rely on it |

The deciding property is the third row. NUTS needs a **gradient**, which needs the density as
a symbolic expression. `pymc_gp`'s is a mixture of Gaussians over its fitted parameters —
differentiable in closed form. An `sbi_npe` flow *has* gradients inside PyTorch, but reaching
them from PyMC's expression system needs an adapter this framework does not include.

### Can the two be installed together?

**Yes, and it is worth being precise, because the history is the opposite of what people
remember.** The two were never mutually incompatible. Until version 0.26, `sbi` listed
`pymc` as a *hard dependency* — so an "SBI environment" always secretly contained PyMC. That
is what once hid a missing PyMC guard in Tutorial 6. From 0.26 onward `pymc` is optional for
`sbi`, so the two are now genuinely **separable and combinable**:

```
py312_bayesmm_all:  pymc 5.26.1 | sbi 0.26.1 | torch 2.13.0   (pip check: no conflicts)
```

The single-backend environments still exist, deliberately, because they mirror CI's
per-backend jobs and their value is in what they *don't* contain. `py312_bayesmm_all` is the
one to use while learning.

### What you will see below, and why it may differ from a colleague's run

The cell is **two-tier**, like the rest of the series:

- **with SBI installed** it fits a small, real `sbi_npe` surrogate (a few seconds) and asks
  the framework about *that*;
- **without SBI** it substitutes a stand-in object and says so.

Both give the same verdict, but only the first is a genuine demonstration. The printout tells
you which you are looking at — so if your output differs from someone else's, check that line
before concluding anything.

### How to choose, in practice

- **Models roughly linear over the range you swept** → `pymc_gp`, and you get the fast
  sampler and proper diagnostics. That is the case in Step 2 deliberately: the physics was
  linear *in logs*, which is common.
- **Strongly non-linear models** → `sbi_npe`, and accept the random-walk sampler. A correct
  answer from a slow sampler beats a fast answer from a model that cannot represent your
  system. Run more draws and **check the ESS**.
- **Unsure** → fit both and compare on held-out sweep points, as T6 does. Fit quality first;
  sampler speed second.


In [ ]:
from bayesian_metamodeling.meta.nuts_sampling import nuts_supported

if not PYMC_AVAILABLE or chain_post is None:
    print("Step 5 SKIPPED — needs Step 2.")
    sbi_was_real = None
else:
    ok, _ = nuts_supported(_chain({}), _surr)
    print(f"all three surrogates are pymc_gp   ->  NUTS available? {ok}")

    # Two-tier: a REAL neural surrogate if sbi is installed, a stand-in otherwise.
    try:
        import sbi  # noqa: F401
        import torch  # noqa: F401
        sbi_was_real = True
    except ImportError:
        sbi_was_real = False

    if sbi_was_real:
        from bayesian_metamodeling.surrogates.backends import fit_backend_model
        _r = np.random.default_rng(3)
        _a = _r.uniform(-2, 2, 120)
        _neural = fit_backend_model(
            backend="sbi_npe", x=_a.reshape(-1, 1),
            y=(1.7*_a + 0.2 + _r.normal(0, 0.05, 120)).reshape(-1, 1),
            input_names=["log_diff"], output_names=["log_decay"],
            backend_config={"density_estimator": "maf", "max_num_epochs": 60,
                            "training_batch_size": 32, "summary_samples": 64},
            seed=0)
        print("  (fitted a REAL sbi_npe surrogate for this comparison)")
    else:
        class _NeuralSurrogate:
            """Stand-in for a fitted sbi_npe model: evaluable, but not differentiable here."""
            def log_prob(self, inputs, outputs):
                raise NotImplementedError
        _neural = _NeuralSurrogate()
        print("  (sbi is not installed, so this uses a STAND-IN, not a real neural surrogate;")
        print("   the verdict is the same, but only a real fit is a genuine demonstration)")

    ok2, reason2 = nuts_supported(_chain({}), {**_surr, "Lck": _neural})
    print(f"swap one for a neural surrogate    ->  NUTS available? {ok2}")
    print(f"  reason: {reason2}")
    print()
    print("From the CLI you would see:")
    print("  --method nuts unavailable (...); falling back to joint.")
    print("You still get an answer, and you are told how it was obtained.")


## Honestly: how this differs from the paper

You have just run the paper's *query shape* on a toy. Before carrying any of these numbers
around, be clear about which parts transfer and which do not.

**What genuinely transfers — the principles:**

- Two measurements at one end of a model chain can inform parameters at the other end, across
  boundaries, in either direction.
- Coupling written as a factor is what makes that possible; written as a generative step it
  would not be.
- Some parameters are identifiable only in combination, and the honest report is the
  combination. The paper's headline `Diff = C·P_off` is exactly this, and Step 3 derived the
  same structure from `λ = √(D/k_off)`.
- Conditioning reaches only what a path of factors connects (Step 4).
- A ridge is where a random walk fails, which is why the sampler and the diagnostics matter.

**What does not transfer — everything specific:**

| | The paper | Here |
|---|---|---|
| the data | real super-resolution microscopy (PALM, IRM) of live T cells | numbers I chose so the answer is checkable |
| the models | a Monte-Carlo membrane simulation, a random-walk Lck model, a spatial pTCR model | three linear maps fitted to synthetic sweeps |
| the surrogates | Bayesian networks with learned conditional distributions | `pymc_gp` linear regressions |
| `Phos_obs`, `Rg_obs` | 22% and 1.31, measured | derived from a truth I planted, so recovery can be verified |
| the finding | a constraint required to reproduce a **peripheral ring of phosphorylated TCRs** seen in real cells | a straight line in a plane, reproducing that constraint's *form* |

**Why a toy is the right teaching tool here.** In Step 2 the truth was known, so "did the
inference work?" had an answer. In a real study it does not — you get a posterior and must
judge it on diagnostics, priors and biological plausibility. Learning to read a ridge on a
problem where you can check the ridge is how you earn the right to read one where you cannot.

**The real thing** is `projects/tcr_signaling/`: four partial models and a metamodel over
them, with `notebooks/03_metamodel_inference.ipynb` running the same commands you just used.
Step 4 above already showed you one honest gap in it — the kinetic-segregation model is not
yet connected, so the paper's `t_KS`/`R_KS` half of the query is not reachable there today.
Finding that with the tools from this notebook, rather than being told it, is roughly the
skill this series exists to build.


## Recap: what module 7 established

- **You can condition any variable on any other**, across model boundaries and in either
  direction, as long as a path of factors connects them. That is the practical payoff of
  building a joint model instead of a pipeline of one-way predictions.
- **Measurements travel upstream.** Two readouts at the end of a chain recovered a hidden
  decay length and a depletion width in models the measurement never touched directly.
- **Some parameters are only identifiable in combination.** When your data sees `D/k_off`,
  report `D/k_off`. The paper's headline result is precisely such a constraint.
- **Check the graph before you run.** A variable with no path to your data will sit at its
  prior forever, and nothing will warn you.
- **Read ESS and r-hat, not just the posterior means** (7b). A ridge is where a random walk
  fails, and a ridge is what coupled models produce.
- **`pymc_gp` and `sbi_npe` are not interchangeable here.** Fit quality decides the backend;
  the backend decides the sampler.

**Where next:** T8 chains three couplings and budgets noise along them. The real
four-surrogate metamodel is
`projects/tcr_signaling/notebooks/03_metamodel_inference.ipynb`.

**The paper:** Neve-Oz, Sherman & Raveh, *Bayesian metamodeling of early T-cell antigen
receptor signaling accounts for its nanoscale activation patterns*, Frontiers in Immunology
15 (2024), [doi:10.3389/fimmu.2024.1412221](https://doi.org/10.3389/fimmu.2024.1412221).
Figure 7 is the query you just ran.


## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `PREFLIGHT: PyMC is missing` | no PyMC in this kernel | `conda env create -f environment-all.yml`, then use the `py312_bayesmm_all` kernel |
| `--method nuts unavailable (... not a diagonal pymc_gp ...)` | a surrogate is `sbi_npe`, or was fitted with `output_correlation: "full"` | Expected — it falls back and still answers. Refit with `pymc_gp` if you want the fast path |
| `r_hat` is `None` | one chain, where r-hat is undefined | use `--chains 2` or more |
| `divergences > 0` | geometry the sampler could not integrate | raise `target_accept`, or look for a coupling `sigma` far tighter than the priors it constrains |
| conditioning barely moves a variable | either no path of factors connects it (Step 6), or the surrogates on that path are weak | inspect the graph first; then check the surrogate's predictive width against the spread of what it predicts |
| a parameter stays at its prior while a *combination* sharpens | non-identifiability (Step 5) | not a bug — report the combination, and design an experiment that separates them |
| inferred values land far outside the training ranges | the observation is not reproducible by the surrogates, so a linear model extrapolates to reach it | check that your measured values are inside what the sweeps actually produced |


## Final check

Asserts what this notebook claims, each in a way that could fail on a run that executed but
demonstrated nothing: NUTS agrees with the closed form and with the random walk; conditioning
moves and narrows; the observed variable is held exactly; the hidden state is recovered
through two model boundaries; the ridge is real (the combination is far sharper than either
parameter); and a non-differentiable surrogate is refused rather than silently mis-sampled.


In [ ]:
if not PYMC_AVAILABLE:
    print("\n[T7c self-check OK] every sampling step skipped per preflight (PyMC absent).")
else:
    # 1. Conditioning moved AND narrowed x, and held y exactly (Step 1), against the
    #    answer worked out on paper.
    assert abs(cs["x"].mean() - post_mean) < 0.06, (
        f"conditioned mean {cs['x'].mean():.4f} != closed form {post_mean:.4f}"
    )
    assert abs(cs["x"].std(ddof=1) - post_sd) < 0.08 * post_sd, "conditioned sd off the closed form"
    assert np.all(cs["y"] == y_measured), "observed variable was sampled instead of clamped"
    assert cs["x"].mean() > nuts_s["x"].mean() + 0.15, "conditioning did not move x"
    assert cs["x"].std() < nuts_s["x"].std(), "a measurement must not widen what it informs"

    # 2. The hidden state was recovered through two model boundaries (Step 2).
    _ld = np.asarray(chain_post["log_decay"], float).ravel()
    _dp = np.asarray(chain_post["depletion"], float).ravel()
    assert abs(_ld.mean() - TRUE_LOG_DECAY) < 0.3, (
        f"log_decay recovered as {_ld.mean():.3f}, truth {TRUE_LOG_DECAY}"
    )
    assert abs(_dp.mean() - TRUE_DEPLETION) < 60, (
        f"depletion recovered as {_dp.mean():.1f}, truth {TRUE_DEPLETION}"
    )
    assert _ld.std() < 0.5 * np.asarray(chain_prior["log_decay"], float).std(), (
        "measuring the readouts should sharply narrow log_decay"
    )
    assert chain_d["divergences"] == 0, f"{chain_d['divergences']} divergences — distrust this"

    # 3. The ridge: the COMBINATION is far better determined than either parameter (Step 3).
    _D = np.asarray(chain_post["log_diff"], float).ravel()
    _K = np.asarray(chain_post["log_poff"], float).ravel()
    _combo_sd = float(np.std(_D - _K))
    assert _combo_sd < 0.5 * min(_D.std(), _K.std()), (
        f"sd(log_diff - log_poff)={_combo_sd:.3f} is not sharper than the individual sds "
        f"({_D.std():.3f}, {_K.std():.3f}) — the identifiability lesson has evaporated"
    )
    assert np.corrcoef(_D, _K)[0, 1] > 0.7, "the posterior is not a ridge"
    # And the ridge sits where the physics says: (log_diff - log_poff)/2 == log lambda.
    assert abs(np.mean(_D - _K)/2 - TRUE_LOG_DECAY) < 0.3, (
        "the ridge is in the wrong place relative to the true decay length"
    )

    # 4. A non-differentiable surrogate is refused, not mis-sampled (Step 5).
    assert not ok2 and "pymc_gp" in reason2, "a neural surrogate must not be claimed NUTS-able"

    _tier = "real sbi_npe" if sbi_was_real else "stand-in (sbi absent)"
    print(f"\n[T7c self-check OK] conditioning {nuts_s['x'].mean():+.2f} -> {cs['x'].mean():+.2f} "
          f"(closed form {post_mean:+.2f}); recovered log_decay={_ld.mean():.2f} (truth "
          f"{TRUE_LOG_DECAY}) and depletion={_dp.mean():.0f} (truth {TRUE_DEPLETION}) through "
          f"two models; ridge sd(combo)={_combo_sd:.2f} vs {_D.std():.2f}/{_K.std():.2f}; "
          f"backend refusal checked with {_tier}; r-hat="
          f"{max(chain_d['r_hat'].values()):.4f}, {chain_d['divergences']} divergences.")
